# 🧠 Continuum SLM — 100M Parameter Conversational AI Training

**Google Colab Training Notebook — Fully Automated**

Simply click **Runtime → Run all** and everything will work automatically!

### What this notebook does:
1. ✅ Clones the project from GitHub
2. ✅ Installs dependencies
3. ✅ Creates Continuum-Max model (102M parameters)
4. ✅ Downloads Alpaca dataset (52K instructions - reliable!)
5. ✅ Trains for 3 epochs (~3-4 hours on free GPU)
6. ✅ Exports quantized model for your phone 📱

---

## ⚙️ Setup

In [ ]:
# ============================================================
# CELL 1: Clone GitHub project & install dependencies
# ============================================================

import os, sys, subprocess

# Clone the project
if not os.path.exists('/content/neuron-ai'):
    !git clone https://github.com/MythroniX24/neuron-ai.git
%cd /content/neuron-ai
sys.path.insert(0, '/content/neuron-ai')

# Install dependencies
!pip install torch numpy tqdm datasets regex flask flask-cors --quiet

print('\n✅ Project cloned and dependencies installed!')
print(f'   Files: {len([f for f in os.listdir(".") if f.endswith(".py")])} Python files')

In [ ]:
# ============================================================
# CELL 2: Check GPU (must enable Runtime -> Change runtime -> GPU)
# ============================================================

import torch

print('=' * 50)
print('GPU CHECK')
print('=' * 50)
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9 if hasattr(props, 'total_memory') else props.total_mem / 1e9
    print(f'VRAM: {vram_gb:.1f} GB')
    print('✅ GPU READY — Training will be FAST!')
else:
    print('⚠️  GPU not detected!')
    print('   Click: Runtime -> Change runtime type -> Hardware accelerator -> GPU')
    print('   Then: Runtime -> Factory reset runtime -> Run all again')
    print('   (CPU will work but be ~10x slower)')

In [ ]:
# ============================================================
# CELL 3: Mount Google Drive for checkpoints
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# Setup directories
CHECKPOINT_DIR = '/content/drive/MyDrive/continuum/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f'\n✅ Checkpoints will be saved to: {CHECKPOINT_DIR}')
print('   (Safe in Google Drive even if Colab session expires)')

## 🏗️ Create Model + Tokenizer

In [ ]:
# ============================================================
# CELL 4: Create 100M Continuum-Max model
# ============================================================

import torch
from continuum.model.model import create_continuum_max

print('Creating Continuum-Max model (~100M parameters)...')
model = create_continuum_max()
print(f'✅ Model created: {model.num_params:,} parameters')

# Move to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

# Quick test
test_input = torch.randint(0, 100, (1, 4)).to(device)
with torch.no_grad():
    out = model.forward(test_input)
print(f'✅ Forward test passed: logits shape {out["logits"].shape}')

# Model info
cfg = model.config
param_mb = model.num_params * 4 / (1024 * 1024)
print(f'\n📊 Model Specs:')
print(f'   Dimensions: d_model={cfg.d_model}, d_state={cfg.d_state}')
print(f'   Layers: {cfg.perception_layers}P + {cfg.core_layers}C + {cfg.output_layers}O = {cfg.n_layers} total')
print(f'   Heads: {cfg.n_heads} Q / {cfg.n_kv_heads} KV (GQA)')
print(f'   Vocab: {cfg.vocab_size} tokens')
print(f'   Memory: {param_mb:.0f} MB (FP32) → {param_mb/4:.0f} MB (INT8 mobile)')

In [ ]:
# ============================================================
# CELL 5: Create tokenizer
# ============================================================

from continuum.tokenizer.bpe import ContinuumTokenizer

tokenizer = ContinuumTokenizer(vocab_size=16000)
print(f'✅ Tokenizer created: {tokenizer.vocab_size_actual} tokens')
print(f'   Special: pad={tokenizer.pad_id}, bos={tokenizer.bos_id}, eos={tokenizer.eos_id}')

# Test
tests = ['Hello, how are you?', 'Namaste! Aap kaise hain?', 'मैं ठीक हूँ']
for t in tests:
    enc = tokenizer.encode_with_special(t)
    dec = tokenizer.decode(enc)
    print(f'   "{t}" → {len(enc)} tokens → "{dec}"')

## 📚 Download & Prepare Dataset

*(Alpaca 52K only - reliable download. Custom data: use load_from_jsonl())*

In [ ]:
# ============================================================
# CELL 6: Download and tokenize datasets
# ============================================================

from continuum.conversation.dataset import ConversationalDataset

print('Downloading and tokenizing datasets...')
print('(Alpaca 52K instructions - reliable! No broken CDN issues)')
print()

dataset = ConversationalDataset(
    tokenizer=tokenizer,
    max_seq_len=1024,
)

# Download ONLY Alpaca (52K public instructions) - reliable Parquet format
# NOTE: Dolly and OpenAssistant are disabled because HuggingFace's CDN
# infrastructure generates broken signed URLs as of 2025-2026.
dataset.load_from_hub(
    include_oasst1=False,
    include_alpaca=True,
    include_dolly=False,
    max_samples=52000,  # Alpaca has 52K, use all
)

stats = dataset.get_stats()
print(f'\n✅ Dataset ready!')
print(f'   Samples: {stats["num_samples"]}')
print(f'   Avg length: {stats["avg_len"]:.0f} tokens')
print(f'   Total tokens: {stats["total_tokens"]:,}')

In [ ]:
# ============================================================
# CELL 7: Create DataLoaders (train/val split)
# ============================================================

import random
from torch.utils.data import DataLoader, Dataset, Subset
from torch.nn.utils.rnn import pad_sequence

# Custom Dataset class
class _Dataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        return (torch.tensor(self.samples[idx][0], dtype=torch.long),
                torch.tensor(self.samples[idx][1], dtype=torch.long))

def collate_fn(batch):
    input_ids = pad_sequence([b[0] for b in batch], batch_first=True, padding_value=0)
    labels = pad_sequence([b[1] for b in batch], batch_first=True, padding_value=-100)
    return {"input_ids": input_ids, "labels": labels}

# Split 90-10
indices = list(range(len(dataset.samples)))
random.shuffle(indices)
val_size = max(1, int(len(indices) * 0.1))

full_dataset = _Dataset(dataset.samples)
train_dataset = Subset(full_dataset, indices[val_size:])
val_dataset = Subset(full_dataset, indices[:val_size])

# ⚡ MEMORY FIX: Small batch size + gradient accumulation
# - batch_size=2 halves activation memory compared to 4
# - gradient_accumulation=4 gives effective batch size of 8
# - The 100M model has token-by-token forward (512 steps of autograd)
#   so each batch is expensive on the 16GB T4
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4  # Effective batch = BATCH_SIZE * GRAD_ACCUM_STEPS = 8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'✅ DataLoaders ready! (Memory-optimized mode)')
print(f'   Train: {len(train_dataset)} samples ({len(train_loader)} batches x {BATCH_SIZE})')
print(f'   Val:   {len(val_dataset)} samples ({len(val_loader)} batches x {BATCH_SIZE})')
print(f'   Gradient accumulation: {GRAD_ACCUM_STEPS} steps → effective batch = {BATCH_SIZE * GRAD_ACCUM_STEPS}')
print(f'\n🚀 Ready to train! Scroll down to "Start Training" cell.')

---
## 🚀 Training

**Training time estimate:**
- GPU (T4): ~3-4 hours for 3 epochs
- CPU only: ~20+ hours (not recommended)

**Colab free session limit:** ~4-6 hours — enough to complete training!

**Sequence-length curriculum:**
- Epoch 1: seq_len=64 (short sequences, low memory)
- Epoch 2: seq_len=256 (medium sequences)
- Epoch 3: seq_len=512 (full length)
This prevents OOM on the first batch.
---

In [ ]:
# ============================================================
# CELL 8: Create trainer
# ============================================================

from continuum.training.trainer import ContinuumTrainer
import math

trainer = ContinuumTrainer(
    model=model,
    learning_rate=3e-4,
    weight_decay=0.1,
    max_grad_norm=1.0,
    warmup_steps=500,
    checkpoint_dir=CHECKPOINT_DIR,
    log_interval=50,
    device=device,
)

print(f'✅ Trainer created')
print(f'   Device: {trainer.device}')
print(f'   Batches per epoch: {len(train_loader)}')

In [ ]:
# ============================================================
# CELL 9: 🚀 START TRAINING!
# ============================================================

import time
import gc

print('=' * 60)
print('🚀 TRAINING STARTED')
print('=' * 60)
print()
print('📊 Legend:')
print('   loss  = Total training loss (should ↓)')
print('   ce    = Cross-entropy loss (should ↓)')
print('   loops = ADL loops used (1-5)')
print('   gamma = GLT decay gate health (0.1-0.9 = healthy)')
print()

NUM_EPOCHS = 3

# Sequence-length curriculum (memory optimization)
# Epoch 1: short sequences → low memory for the massive autograd graph
# Epoch 2: medium sequences
# Epoch 3: full-length sequences
SEQ_LEN_CURRICULUM = [64, 256, 512]

# Track losses for plotting
train_losses = []
val_losses = []
val_ppls = []
epochs_done = []

for epoch in range(NUM_EPOCHS):
    seq_len = SEQ_LEN_CURRICULUM[epoch]
    
    # Clear GPU cache before each epoch (prevents fragmentation)
    if device == 'cuda':
        torch.cuda.empty_cache()
        gc.collect()
    
    epoch_start = time.time()
    epoch_loss = 0.0
    epoch_steps = 0
    
    from tqdm import tqdm
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} (L={seq_len})')
    
    # Training with gradient accumulation
    trainer.model.train()
    accum_loss = 0.0
    accum_steps = 0
    
    for batch_idx, batch in enumerate(pbar):
        accum_idx = (batch_idx % GRAD_ACCUM_STEPS) + 1
        
        # Forward + backward (accumulates gradients internally)
        metrics = trainer.train_step(
            batch, 
            seq_len_curriculum=seq_len,
            accumulation_step=accum_idx,
            total_accumulation_steps=GRAD_ACCUM_STEPS,
        )
        
        epoch_loss += metrics['loss']
        epoch_steps += 1
        accum_loss += metrics['loss']
        accum_steps += 1
        
        # Log after each optimizer step (i.e., after every GRAD_ACCUM_STEPS batches)
        if accum_idx == GRAD_ACCUM_STEPS and trainer.global_step % 50 == 0:
            pbar.set_postfix({
                'loss': f"{accum_loss/accum_steps:.3f}",
                'ce': f"{metrics['ce_loss']:.3f}",
                'loops': f"{metrics['n_loops']:.1f}",
                'gamma': f"{metrics.get('gamma_mean', 0):.2f}",
            })
            accum_loss = 0.0
            accum_steps = 0
    
    # End of epoch
    avg_loss = epoch_loss / max(epoch_steps, 1)
    train_losses.append(avg_loss)
    epochs_done.append(epoch + 1)
    
    # Clear cache before validation
    if device == 'cuda':
        torch.cuda.empty_cache()
        gc.collect()
    
    # Validation
    val_metrics = trainer.validate(val_loader)
    val_losses.append(val_metrics['val_loss'])
    val_ppls.append(val_metrics['val_ppl'])
    
    epoch_time = time.time() - epoch_start
    
    print(f'\n📊 Epoch {epoch+1} results:')
    print(f'   Loss:     {avg_loss:.4f}  (train) / {val_metrics["val_loss"]:.4f}  (val)')
    print(f'   Perplexity: {val_metrics["val_ppl"]:.1f}')
    print(f'   Time:     {epoch_time:.0f}s')
    print()
    
    # Save checkpoint
    trainer.save_checkpoint(epoch, is_best=(val_metrics['val_loss'] < trainer.best_val_loss))

print('=' * 60)
print('✅ TRAINING COMPLETE!')
print('=' * 60)

In [ ]:
# ============================================================
# CELL 10: Plot training curves
# ============================================================

try:
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(15, 4))
    
    # Loss
    plt.subplot(1, 3, 1)
    plt.plot(epochs_done, train_losses, 'b-o', label='Train Loss')
    plt.plot(epochs_done, val_losses, 'r-o', label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training & Validation Loss')
    plt.legend()
    plt.grid(True)
    
    # Perplexity
    plt.subplot(1, 3, 2)
    plt.plot(epochs_done, val_ppls, 'g-o')
    plt.xlabel('Epoch')
    plt.ylabel('Perplexity')
    plt.title('Validation Perplexity')
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Summary
    print('📈 Training Summary:')
    for i in range(len(epochs_done)):
        print(f'   Epoch {epochs_done[i]}: loss={train_losses[i]:.4f} → val_loss={val_losses[i]:.4f}, ppl={val_ppls[i]:.1f}')
    
except ImportError:
    print('📊 Loss values:')
    for i in range(len(epochs_done)):
        print(f'   Epoch {epochs_done[i]}: train_loss={train_losses[i]:.4f}, val_loss={val_losses[i]:.4f}, ppl={val_ppls[i]:.1f}')

## 🧪 Test the Model

In [ ]:
# ============================================================
# CELL 11: Test conversation generation
# ============================================================

from continuum.conversation.manager import ConversationManager

# Create manager
manager = ConversationManager(
    model=model,
    tokenizer=tokenizer,
    device='cpu',
    quantize=False,
)

# Test prompts
test_prompts = [
    'What is the capital of France?',
    'Explain quantum computing in simple terms.',
    'Write a short poem about technology.',
]

for prompt in test_prompts:
    print(f'\n{"="*60}')
    print(f'👤 User: {prompt}')
    response = manager.chat(prompt, max_new_tokens=100)
    print(f'🤖 Assistant: {response}')
    manager.new_conversation()

In [ ]:
# ============================================================
# CELL 12: Multi-turn conversation test
# ============================================================

manager.new_conversation()

turns = [
    'Hi! My name is Alex.',
    'What is my name?',
    'Can you tell me a fun fact?',
]

for turn in turns:
    print(f'\n👤 User: {turn}')
    response = manager.chat(turn, max_new_tokens=100)
    print(f'🤖 Assistant: {response}')

print(f'\n✅ Held {len(manager.conversation.history)} turns of conversation')

## 📱 Export for Mobile Phone

After this step, download the files from Google Drive to your phone.

In [ ]:
# ============================================================
# CELL 13: Export trained model for phone
# ============================================================

# Save full model
export_path = os.path.join(CHECKPOINT_DIR, 'continuum_max_for_mobile.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'config': model.config,
    'num_params': model.num_params,
}, export_path)

size_mb = os.path.getsize(export_path) / (1024 * 1024)
print(f'✅ Model exported: {export_path}')
print(f'   Size: {size_mb:.1f} MB (FP32)')
print(f'   Phone size (INT8): {size_mb/4:.1f} MB')

# Save tokenizer
tokenizer_path = os.path.join(CHECKPOINT_DIR, 'tokenizer_16k.json')
tokenizer.save(tokenizer_path)
print(f'✅ Tokenizer saved: {tokenizer_path}')

In [ ]:
# ============================================================
# CELL 14: Test INT8 quantized model (phone-ready)
# ============================================================

from continuum.inference.engine import ContinuumInference

print('Testing INT8 quantized inference...')

engine = ContinuumInference(
    model=model,
    tokenizer=tokenizer,
    device='cpu',
    quantize=True,
)

stats = engine.get_stats()
print(f'📱 Quantized model: {stats["model_size_mb"]:.1f} MB')
print(f'   State checkpoint: {stats.get("checkpoint_kb", 0):.0f} KB')
print(f'   Ready for phone deployment!')

# Test
test_text = 'Namaste! Aap kaise hain?'
print(f'\n👤 User: {test_text}')
response = engine.generate(
    test_text, max_new_tokens=50, temperature=0.8, stream=False
)
print(f'🤖 Assistant: {response}')

---
## ✅ Training Complete!

### 📱 How to use on your phone:

**1. Download files from Google Drive:**
```
MyDrive/continuum/checkpoints/
├── continuum_max_for_mobile.pt    ← Download this (~400 MB)
├── tokenizer_16k.json              ← Download this
└── checkpoint_epoch_*.pt          ← Extra backups
```

**2. Copy to phone:**
```bash
# On your Android phone terminal (Termux/Proot):
cp /sdcard/Download/continuum_max_for_mobile.pt /mnt/sdcard/Download/Neuron/checkpoints/
cp /sdcard/Download/tokenizer_16k.json /mnt/sdcard/Download/Neuron/checkpoints/
```

**3. Start chat server on phone:**
```bash
cd /mnt/sdcard/Download/Neuron
python3 continuum/run.py chat --model checkpoints/continuum_max_for_mobile.pt --host 0.0.0.0 --port 5000
```

**4. Open in phone browser:** `http://localhost:5000`

### ✨ Custom Data?
Want to add your own data? Create a JSONL file and load it:
```python
dataset.load_from_jsonl("my_custom_data.jsonl")  # Add after Alpaca
```

---

> **Tip:** If Colab session expires, restart with:
> 1. Runtime → Factory reset runtime
> 2. Run all cells again
> 3. Cell 9 will auto-load from Drive: `torch.load(CHECKPOINT_DIR + '/best_model.pt')`

---
🎉 **Congratulations! You've trained your own 100M parameter AI!**